# Dijkstra's Algorithm: Step-by-Step Trace

**Indian Railways freight network | Graph Theory + Python**

This companion notebook traces Dijkstra's shortest-path algorithm on the railway dataset. It shows which junction is settled at each iteration, which tentative distances change, how predecessors build the route, and how the final result compares with NetworkX.

Run the cells from top to bottom. Keep `indian_railways.csv` beside this notebook. Distances are the edge weights in this trace, so the result is the shortest route by kilometres.

## 1. What Dijkstra's algorithm does

Dijkstra's algorithm finds minimum-weight paths from one source node when every edge weight is non-negative. It maintains:

- a **tentative distance** for each junction (initially infinity, except the source at zero);
- a **predecessor** for each junction, recording the previous junction on its current best-known route;
- a set of **unsettled** junctions that have not yet had their shortest distance finalized.

At each iteration, select the unsettled junction with the smallest tentative distance and settle it. Then inspect each unsettled neighbour. If travelling through the selected junction gives the neighbour a shorter distance, update the distance and predecessor. Because weights are non-negative, a settled distance cannot later be improved.

Here, an edge's weight is its `Distance_km`. Estimated travel time is not used in this trace.

## 2. Load the data and build the graph

Each CSV row becomes an undirected graph edge between its origin and destination. The distance is stored as the weight used by Dijkstra. This is the same network representation used in the main project notebook.

In [1]:
from pathlib import Path

import networkx as nx
import pandas as pd

DATA_PATH = Path("indian_railways.csv")
required_columns = {"Origin", "Destination", "Distance_km"}

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Keep the CSV beside this notebook "
        "and run Jupyter from the project folder."
    )

df = pd.read_csv(DATA_PATH)
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing columns: {sorted(missing_columns)}")
if df[["Origin", "Destination", "Distance_km"]].isna().any().any():
    raise ValueError("Origin, Destination, and Distance_km cannot be missing.")
if (df["Distance_km"] <= 0).any():
    raise ValueError("Dijkstra's distance weights must be positive.")

G = nx.Graph()
for row in df.itertuples(index=False):
    G.add_edge(row.Origin, row.Destination, distance=float(row.Distance_km))

print(f"Loaded {len(df)} sections, {G.number_of_nodes()} junctions, "
      f"and {G.number_of_edges()} graph edges.")
print(f"Network connected: {nx.is_connected(G)}")


Loaded 51 sections, 44 junctions, and 51 graph edges.
Network connected: True


## 3. Choose the source and destination

We will find the minimum-distance route from New Delhi to Chennai. The graph is undirected, so its sections can be traversed in either direction in this simplified model.

In [2]:
SOURCE = "New Delhi"
DESTINATION = "Chennai"

for junction in (SOURCE, DESTINATION):
    if junction not in G:
        raise ValueError(f"Junction not found in the dataset: {junction}")

print(f"Find the shortest-distance route from {SOURCE} to {DESTINATION}.")


Find the shortest-distance route from New Delhi to Chennai.


## 4. Initialize the algorithm

Before the first iteration, only the source has a finite tentative distance: zero. Every other junction is initially unreachable from the information discovered so far, so its tentative distance is infinity. No predecessor has been assigned.

In [3]:
distances = {junction: float("inf") for junction in G.nodes}
predecessors = {junction: None for junction in G.nodes}
unsettled = set(G.nodes)
distances[SOURCE] = 0

pd.DataFrame(
    [(junction, distances[junction], predecessors[junction])
     for junction in sorted(G.nodes)],
    columns=["Junction", "Tentative distance (km)", "Predecessor"],
).head(10)


,Junction,Tentative distance (km),Predecessor
0,Agra Cantt,inf,None
1,Asansol,inf,None
2,Balharshah,inf,None
3,Bengaluru,inf,None
4,Bhopal,inf,None
5,Bhubaneswar,inf,None
6,Bilaspur,inf,None
7,Chennai,inf,None
8,Dhanbad,inf,None
9,Durg,inf,None


## 5. Run and record every iteration

The function below implements the algorithm directly rather than calling NetworkX's shortest-path routine. The selected junction is the unsettled node with the smallest `(distance, name)` pair; the name is used only to make equal-distance ties reproducible. Every successful relaxation is recorded with its previous and new distance, and the algorithm stops once the destination is settled.

In [4]:
def trace_dijkstra(graph, source, destination):
    """Run Dijkstra and return iteration and distance-update logs."""
    distances = {junction: float("inf") for junction in graph.nodes}
    predecessors = {junction: None for junction in graph.nodes}
    unsettled = set(graph.nodes)
    distances[source] = 0.0
    iteration_log = []
    update_log = []
    step = 0

    while unsettled:
        current = min(unsettled, key=lambda node: (distances[node], node))
        if distances[current] == float("inf"):
            break

        unsettled.remove(current)
        step += 1
        updates_this_step = []

        for neighbour in sorted(graph.neighbors(current)):
            if neighbour not in unsettled:
                continue

            edge_distance = graph[current][neighbour]["distance"]
            candidate = distances[current] + edge_distance
            previous = distances[neighbour]

            if candidate < previous:
                distances[neighbour] = candidate
                predecessors[neighbour] = current
                updates_this_step.append(
                    f"{neighbour}: {previous:g} -> {candidate:g} km"
                )
                update_log.append({
                    "Step": step,
                    "Settled junction": current,
                    "Neighbour": neighbour,
                    "Previous distance (km)": previous,
                    "Candidate distance (km)": candidate,
                    "Decision": "Updated",
                    "New predecessor": current,
                })
            else:
                update_log.append({
                    "Step": step,
                    "Settled junction": current,
                    "Neighbour": neighbour,
                    "Previous distance (km)": previous,
                    "Candidate distance (km)": candidate,
                    "Decision": "Kept existing distance",
                    "New predecessor": predecessors[neighbour],
                })

        iteration_log.append({
            "Step": step,
            "Settled junction": current,
            "Settled distance (km)": distances[current],
            "Distance updates": "; ".join(updates_this_step) or "No updates",
        })

        if current == destination:
            break

    if distances[destination] == float("inf"):
        raise nx.NetworkXNoPath(f"No route from {source} to {destination}.")

    return distances, predecessors, iteration_log, update_log


distances, predecessors, iteration_log, update_log = trace_dijkstra(
    G, SOURCE, DESTINATION
)
print(f"Destination settled after {len(iteration_log)} iterations.")


Destination settled after 41 iterations.


## 6. Inspect the settled-junction trace

Each row is one iteration. The settled distance is now final. The updates column lists neighbours whose tentative distances improved by travelling through that junction. Initially, the source is settled at zero and its direct neighbours receive their section distances.

In [5]:
iteration_table = pd.DataFrame(iteration_log)
iteration_table


,Step,Settled junction,Settled distance (km),Distance updates
0,1,New Delhi,0.0,Agra Cantt: inf -> 195 km; Kanpur: inf -> 440 ...
1,2,Agra Cantt,195.0,Gwalior: inf -> 313 km
2,3,Gwalior,313.0,Jhansi: inf -> 410 km
3,4,Jhansi,410.0,Bhopal: inf -> 701 km
4,5,Kanpur,440.0,Prayagraj: inf -> 635 km
5,6,Kota,465.0,Ratlam: inf -> 731 km
6,7,Prayagraj,635.0,Katni: inf -> 911 km; Pt Deen Dayal Upadhyaya:...
7,8,Bhopal,701.0,Itarsi: inf -> 793 km; Ujjain: inf -> 885 km
8,9,Ratlam,731.0,Ujjain: 885 -> 828 km; Vadodara: inf -> 992 km
9,10,Pt Deen Dayal Upadhyaya,788.0,Gaya: inf -> 993 km; Patna: inf -> 1000 km


## 7. Inspect each distance relaxation

A **relaxation** tests whether a neighbour can be reached more cheaply through the current junction. `Updated` means the candidate was smaller, so both distance and predecessor changed. `Kept existing distance` means the neighbour already had an equal or shorter known route. Sorting by iteration preserves the order of the trace.

In [6]:
relaxation_table = pd.DataFrame(update_log)
relaxation_table


,Step,Settled junction,Neighbour,Previous distance (km),Candidate distance (km),Decision,New predecessor
0,1,New Delhi,Agra Cantt,inf,195.0,Updated,New Delhi
1,1,New Delhi,Kanpur,inf,440.0,Updated,New Delhi
2,1,New Delhi,Kota,inf,465.0,Updated,New Delhi
3,2,Agra Cantt,Gwalior,inf,313.0,Updated,Agra Cantt
4,3,Gwalior,Jhansi,inf,410.0,Updated,Gwalior
5,4,Jhansi,Bhopal,inf,701.0,Updated,Jhansi
6,5,Kanpur,Prayagraj,inf,635.0,Updated,Kanpur
7,6,Kota,Ratlam,inf,731.0,Updated,Kota
8,7,Prayagraj,Katni,inf,911.0,Updated,Prayagraj
9,7,Prayagraj,Pt Deen Dayal Upadhyaya,inf,788.0,Updated,Prayagraj


## 8. Reconstruct the route from predecessors

Once the destination is settled, follow predecessor links backward until reaching the source, then reverse that list. This route is not stored as a complete path at every iteration; it is recovered from the predecessor map.

In [7]:
path = []
current = DESTINATION
while current is not None:
    path.append(current)
    if current == SOURCE:
        break
    current = predecessors[current]

path.reverse()
if not path or path[0] != SOURCE:
    raise RuntimeError("Could not reconstruct a route back to the source.")

path_distance = sum(
    G[u][v]["distance"] for u, v in zip(path, path[1:])
)
print(" -> ".join(path))
print(f"Total distance: {path_distance:.0f} km")
print(f"Final tentative distance at destination: {distances[DESTINATION]:.0f} km")
assert path_distance == distances[DESTINATION]


New Delhi -> Agra Cantt -> Gwalior -> Jhansi -> Bhopal -> Itarsi -> Nagpur -> Balharshah -> Kazipet -> Vijayawada -> Chennai
Total distance: 2181 km
Final tentative distance at destination: 2181 km


## 9. Verify with NetworkX

Finally, compare the hand-traced implementation with NetworkX's Dijkstra implementation using the same graph and distance weight. Matching path lengths verifies the computed minimum distance. If multiple equal-length routes exist, the exact node sequence can differ while both paths remain correct.

In [8]:
networkx_path = nx.dijkstra_path(G, SOURCE, DESTINATION, weight="distance")
networkx_distance = nx.dijkstra_path_length(
    G, SOURCE, DESTINATION, weight="distance"
)

print("Manual trace distance:", f"{path_distance:.0f} km")
print("NetworkX distance:     ", f"{networkx_distance:.0f} km")
print("Manual trace route:    ", " -> ".join(path))
print("NetworkX route:        ", " -> ".join(networkx_path))
assert path_distance == networkx_distance
print("\nVerification passed: both methods find the same shortest distance.")


Manual trace distance: 2181 km
NetworkX distance:      2181 km
Manual trace route:     New Delhi -> Agra Cantt -> Gwalior -> Jhansi -> Bhopal -> Itarsi -> Nagpur -> Balharshah -> Kazipet -> Vijayawada -> Chennai
NetworkX route:         New Delhi -> Agra Cantt -> Gwalior -> Jhansi -> Bhopal -> Itarsi -> Nagpur -> Balharshah -> Kazipet -> Vijayawada -> Chennai

Verification passed: both methods find the same shortest distance.


## 10. Key takeaways and limitations

- The algorithm repeatedly finalizes the currently closest unsettled junction and improves tentative routes to its neighbours.
- The predecessor map is enough to reconstruct the final shortest route.
- This trace uses distance as the edge weight. Use a separate time-weighted run to answer a fastest-route question.
- The railway graph and distances are a simplified educational dataset. The result is not a live timetable and excludes operational delays.

For a larger graph, a priority queue is more efficient than scanning all unsettled nodes for the minimum on every iteration. This transparent version uses the scan so the selection rule is easy to follow.